In [16]:
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [17]:
class LLMState(TypedDict):
    topic:str
    outline:str
    blog:str

In [18]:
load_dotenv()
llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")

In [23]:
def create_outline(state:LLMState)->LLMState:
     # Extract Topic
    topic=state['topic']
    
    # Form PromptTemplate
    prompt=PromptTemplate(
        template="""Generate me Outline for the following topic make it crisp,short but tell complete picture \n topic:{topic} """,
        input_variables=['topic']
        )
    
    # Define Parser
    parser=StrOutputParser()
    
    # Forming Chain
    chain=prompt | llm | parser 
    
    # Call LLM
    outline=chain.invoke({'topic':topic})
    
    # Update Answer
    state['outline']=outline
    
    # Return State
    return state  

In [25]:
def create_blog(state:LLMState)->LLMState:
     # Extract Outline
    outline=state['outline']
    
    # Form PromptTemplate
    prompt=PromptTemplate(
        template="""Generate me Blog From The Following Outline it should be crisp and easy to understand it must not be too long \n topic:{outline} """,
        input_variables=['outline']
        )
    
    # Define Parser
    parser=StrOutputParser()
    
    # Forming Chain
    chain=prompt | llm | parser 
    
    # Call LLM
    blog=chain.invoke({'outline':outline})
    
    # Update Answer
    state['blog']=blog
    
    # Return State
    return state  

In [21]:
# Define Graph
graph=StateGraph(LLMState)

# Add Nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

# Add Edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)

# Compile Graph
workflow=graph.compile()


In [22]:
initial_state={'topic':'Fear Of AI Nowdays?'}

final_output=workflow.invoke(initial_state)
print(final_output)

{'topic': 'Fear Of AI Nowdays?', 'outline': 'Here is a crisp, comprehensive outline regarding the current fear of AI.\n\n### **Topic: The Anatomy of AI Anxiety**\n\n**I. The Core Drivers of Fear**\n*   **Economic Displacement:** Anxiety regarding automation-led job losses across white-collar and creative sectors.\n*   **Existential Risk:** Fears surrounding "Superintelligence" and the loss of human control (The Alignment Problem).\n*   **Erosion of Truth:** The crisis of "Deepfakes," misinformation, and the inability to distinguish reality from synthetic content.\n\n**II. The "Black Box" Problem**\n*   **Lack of Transparency:** Fear rooted in the inability to understand how AI models reach specific decisions (algorithmic opacity).\n*   **Bias and Inequality:** Concerns that AI reinforces historical prejudices and social stratification at an automated scale.\n\n**III. Socio-Psychological Impact**\n*   **Dependency:** The fear of "intellectual atrophy"—losing critical thinking and creati